# VVEJSA Trajectory Search
## Venus-Venus-Earth-Jupiter-Saturn Gravity Assist

Searches for viable VVEJSA trajectories over a launch window using global optimization.
The optimizer varies launch date and time-of-flight for each leg, minimizing launch C3
while enforcing flyby feasibility constraints (altitude above surface, v-infinity matching).

In [ ]:
from astropy import units as u
from astropy import time
import numpy as np

from poliastro.bodies import Earth, Venus, Jupiter, Saturn, Sun
from poliastro.ephem import Ephem
from poliastro.maneuver import Maneuver
from poliastro.twobody import Orbit
from poliastro.util import time_range
from poliastro.frames import Planes
from copy import deepcopy

from poliastro.plotting import StaticOrbitPlotter
import matplotlib.pyplot as plt
import plotly.io as pio
from poliastro.util import norm
pio.renderers.default = "plotly_mimetype+notebook_connected"
from astropy.coordinates import solar_system_ephemeris
import warnings

solar_system_ephemeris.set("jpl")

## Search Configuration
Define the launch window and TOF bounds for each leg. Ephemerides are generated
to cover the full possible date range.

In [ ]:
# === SEARCH PARAMETERS ===
# Launch window
launch_window_start = "1996-01-01"
launch_window_end   = "1999-01-01"

# TOF bounds and step sizes for each leg (days)
#                       min,  max,  step
tof_config = {
    "Earth→Venus1":    (100,  350,  10),
    "Venus1→Venus2":   (200,  500,  10),
    "Venus2→Earth":    (30,   200,   7),
    "Earth→Jupiter":   (300,  900,  15),
    "Jupiter→Saturn":  (500, 2000,  30),
}

# Launch date step (days)
launch_step = 15

# Max v-infinity mismatch at a flyby to consider it viable (km/s, per flyby)
# Allows some powered flyby delta-v — tighter = fewer results but more realistic
max_vinf_mismatch = 0.5  # km/s (500 m/s) — per flyby

# Minimum flyby altitudes (km)
min_flyby_alt = {
    Venus: 300,
    Earth: 300,
    Jupiter: 50000,
}

# Sequence definition
flyby_bodies = [Earth, Venus, Venus, Earth, Jupiter, Saturn]
leg_names = list(tof_config.keys())

# Reference epochs
launch_start_jd = time.Time(launch_window_start, scale="utc").tdb.jd
launch_end_jd = time.Time(launch_window_end, scale="utc").tdb.jd

# Ephemeris date range
max_total_tof = sum(cfg[1] for cfg in tof_config.values())
ephem_start = time.Time(launch_window_start, scale="utc").tdb - 100 * u.day
ephem_end = time.Time(launch_window_end, scale="utc").tdb + max_total_tof * u.day + 100 * u.day

# Print search space size
n_launch = int((launch_end_jd - launch_start_jd) / launch_step) + 1
total_grid = n_launch
print(f"Launch window: {launch_window_start} to {launch_window_end} ({n_launch} dates, step={launch_step}d)")
print(f"\nTOF grid per leg:")
for name, (lo, hi, step) in tof_config.items():
    n = int((hi - lo) / step) + 1
    total_grid *= n
    print(f"  {name:18s}: {lo:5d} - {hi:5d} d, step={step:2d}d ({n:3d} points)")
print(f"\nFull grid (without pruning): {total_grid:,} evaluations")
print(f"With forward pruning this will be MUCH smaller.")

In [ ]:
# Generate ephemerides covering the full search range
ephem_venus = Ephem.from_body(Venus, time_range(ephem_start, end=ephem_end, periods=3000), plane=Planes.EARTH_ECLIPTIC)
ephem_earth = Ephem.from_body(Earth, time_range(ephem_start, end=ephem_end, periods=3000), plane=Planes.EARTH_ECLIPTIC)
ephem_jupiter = Ephem.from_body(Jupiter, time_range(ephem_start, end=ephem_end, periods=3000), plane=Planes.EARTH_ECLIPTIC)
ephem_saturn = Ephem.from_body(Saturn, time_range(ephem_start, end=ephem_end, periods=3000), plane=Planes.EARTH_ECLIPTIC)

ephem_map = {
    Venus: ephem_venus,
    Earth: ephem_earth,
    Jupiter: ephem_jupiter,
    Saturn: ephem_saturn,
}

print("Ephemerides generated successfully.")

## Helper Functions
Lambert solver wrapper and flyby feasibility checker.

In [ ]:
def solve_leg(dep_body, arr_body, dep_date, arr_date):
    """
    Solve Lambert for one leg. Returns (transfer_orb, v_dep, v_arr) or None on failure.
    v_dep/v_arr are heliocentric velocities of the transfer orbit at departure/arrival.
    """
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            dep_orb = Orbit.from_ephem(Sun, ephem_map[dep_body], dep_date)
            arr_orb = Orbit.from_ephem(Sun, ephem_map[arr_body], arr_date)
            man = Maneuver.lambert(dep_orb, arr_orb)
            trans = dep_orb.apply_maneuver(man, intermediate=True)[0]
            v_dep = trans.rv()[1]
            trans_arr = trans.propagate(arr_date - dep_date)
            v_arr = trans_arr.rv()[1]
            return trans, v_dep, v_arr
    except Exception:
        return None


def check_flyby(body, flyby_date, v_arr_helio, v_dep_helio):
    """
    Check flyby feasibility. Returns (vinf_mismatch, altitude_km, turn_deg) or None.
    """
    try:
        body_orb = Orbit.from_ephem(Sun, ephem_map[body], flyby_date)
        body_vel = body_orb.rv()[1]
        
        v_inf_in = v_arr_helio - body_vel
        v_inf_out = v_dep_helio - body_vel
        
        v_inf_in_mag = np.linalg.norm(v_inf_in).to(u.km / u.s).value
        v_inf_out_mag = np.linalg.norm(v_inf_out).to(u.km / u.s).value
        
        vinf_mismatch = abs(v_inf_out_mag - v_inf_in_mag)
        
        cos_turn = np.dot(v_inf_in.value, v_inf_out.value) / (
            np.linalg.norm(v_inf_in.value) * np.linalg.norm(v_inf_out.value))
        cos_turn = np.clip(cos_turn, -1, 1)
        turn_angle = np.arccos(cos_turn)
        
        v_inf_avg = (v_inf_in_mag + v_inf_out_mag) / 2
        sin_half = np.sin(turn_angle / 2)
        if sin_half < 1e-10:
            return None
        
        mu = body.k.to(u.km**3 / u.s**2).value
        r_p = mu / (v_inf_avg**2) * (1/sin_half - 1)
        altitude = r_p - body.R.to(u.km).value
        
        return vinf_mismatch, altitude, np.degrees(turn_angle)
    except Exception:
        return None

print("Helper functions defined.")

## Forward-Chaining Grid Search
Searches leg-by-leg, pruning infeasible branches at each flyby.
Only evaluates downstream legs if the current flyby is feasible.

In [ ]:
# Build TOF arrays for each leg
tof_arrays = []
for name, (lo, hi, step) in tof_config.items():
    tof_arrays.append(np.arange(lo, hi + 1, step))

launch_jds = np.arange(launch_start_jd, launch_end_jd + 1, launch_step)

# Track all viable complete trajectories
# Each entry: (score, [launch_jd, tof1, tof2, tof3, tof4, tof5], total_vinf_mismatch)
viable_trajectories = []

# Counters for progress
counts = {i: 0 for i in range(6)}  # how many candidates survive each leg
total_lamberts = 0

print(f"Searching {len(launch_jds)} launch dates × {len(tof_arrays[0])} TOF1 values...")
print("Pruning at each flyby for altitude and v∞ mismatch.\n")

for li, ljd in enumerate(launch_jds):
    launch_date = time.Time(ljd, format="jd", scale="tdb")
    
    if li % 10 == 0:
        print(f"  Launch date {li+1}/{len(launch_jds)}: {launch_date.iso[:10]}  "
              f"(viable so far: {len(viable_trajectories)})")
    
    for tof1 in tof_arrays[0]:
        # Leg 1: Earth → Venus1
        v1_date = launch_date + tof1 * u.day
        leg1 = solve_leg(Earth, Venus, launch_date, v1_date)
        total_lamberts += 1
        if leg1 is None:
            continue
        trans1, v1_dep, v1_arr = leg1
        counts[0] += 1
        
        for tof2 in tof_arrays[1]:
            # Leg 2: Venus1 → Venus2
            v2_date = v1_date + tof2 * u.day
            leg2 = solve_leg(Venus, Venus, v1_date, v2_date)
            total_lamberts += 1
            if leg2 is None:
                continue
            trans2, v2_dep, v2_arr = leg2
            
            # Check Venus 1 flyby
            fb1 = check_flyby(Venus, v1_date, v1_arr, v2_dep)
            if fb1 is None:
                continue
            mismatch1, alt1, turn1 = fb1
            if mismatch1 > max_vinf_mismatch or alt1 < min_flyby_alt[Venus]:
                continue
            counts[1] += 1
            
            for tof3 in tof_arrays[2]:
                # Leg 3: Venus2 → Earth
                e_date = v2_date + tof3 * u.day
                leg3 = solve_leg(Venus, Earth, v2_date, e_date)
                total_lamberts += 1
                if leg3 is None:
                    continue
                trans3, v3_dep, v3_arr = leg3
                
                # Check Venus 2 flyby
                fb2 = check_flyby(Venus, v2_date, v2_arr, v3_dep)
                if fb2 is None:
                    continue
                mismatch2, alt2, turn2 = fb2
                if mismatch2 > max_vinf_mismatch or alt2 < min_flyby_alt[Venus]:
                    continue
                counts[2] += 1
                
                for tof4 in tof_arrays[3]:
                    # Leg 4: Earth → Jupiter
                    j_date = e_date + tof4 * u.day
                    leg4 = solve_leg(Earth, Jupiter, e_date, j_date)
                    total_lamberts += 1
                    if leg4 is None:
                        continue
                    trans4, v4_dep, v4_arr = leg4
                    
                    # Check Earth flyby
                    fb3 = check_flyby(Earth, e_date, v3_arr, v4_dep)
                    if fb3 is None:
                        continue
                    mismatch3, alt3, turn3 = fb3
                    if mismatch3 > max_vinf_mismatch or alt3 < min_flyby_alt[Earth]:
                        continue
                    counts[3] += 1
                    
                    for tof5 in tof_arrays[4]:
                        # Leg 5: Jupiter → Saturn
                        s_date = j_date + tof5 * u.day
                        leg5 = solve_leg(Jupiter, Saturn, j_date, s_date)
                        total_lamberts += 1
                        if leg5 is None:
                            continue
                        trans5, v5_dep, v5_arr = leg5
                        
                        # Check Jupiter flyby
                        fb4 = check_flyby(Jupiter, j_date, v4_arr, v5_dep)
                        if fb4 is None:
                            continue
                        mismatch4, alt4, turn4 = fb4
                        if mismatch4 > max_vinf_mismatch or alt4 < min_flyby_alt[Jupiter]:
                            continue
                        counts[4] += 1
                        
                        # Compute launch C3
                        earth_orb = Orbit.from_ephem(Sun, ephem_map[Earth], launch_date)
                        v_earth = earth_orb.rv()[1]
                        v_inf_launch = np.linalg.norm(v1_dep - v_earth).to(u.km/u.s).value
                        c3 = v_inf_launch ** 2
                        
                        total_mismatch = mismatch1 + mismatch2 + mismatch3 + mismatch4
                        score = c3 + (total_mismatch * 100) ** 2
                        
                        viable_trajectories.append((
                            score,
                            [ljd, tof1, tof2, tof3, tof4, tof5],
                            c3,
                            total_mismatch,
                        ))
                        counts[5] += 1

print(f"\n{'='*60}")
print(f"Search complete!")
print(f"Total Lambert solves: {total_lamberts:,}")
print(f"Candidates surviving each stage:")
stage_names = ["Leg1 OK", "Venus1 FB", "Venus2 FB", "Earth FB", "Jupiter FB", "Complete"]
for i, name in enumerate(stage_names):
    print(f"  {name:12s}: {counts[i]:,}")
print(f"\nFound {len(viable_trajectories)} viable VVEJSA trajectories!")

## Best Trajectories
Sort by score and display the top results.

In [ ]:
if not viable_trajectories:
    print("No viable trajectories found! Try relaxing constraints:")
    print("  - Increase max_vinf_mismatch")
    print("  - Decrease min_flyby_alt values")
    print("  - Widen TOF bounds or launch window")
else:
    # Sort by score (lower is better)
    viable_trajectories.sort(key=lambda x: x[0])
    
    print(f"Top {min(10, len(viable_trajectories))} trajectories (sorted by score):")
    print(f"{'#':>3s}  {'Score':>10s}  {'C3 km²/s²':>10s}  {'ΔV∞ tot':>8s}  "
          f"{'Launch':>12s}  {'TOFs (days)':>40s}")
    print("-" * 95)
    
    for rank, (score, params, c3, mismatch) in enumerate(viable_trajectories[:10]):
        ljd = params[0]
        tofs = params[1:]
        launch_iso = time.Time(ljd, format="jd", scale="tdb").iso[:10]
        tof_str = ", ".join(f"{t:.0f}" for t in tofs)
        print(f"{rank+1:3d}  {score:10.1f}  {c3:10.2f}  {mismatch:7.2f}  "
              f"{launch_iso:>12s}  [{tof_str}]")
    
    # Select best trajectory
    best_score, best_params, best_c3, best_mismatch = viable_trajectories[0]
    best_launch_jd = best_params[0]
    best_tofs = best_params[1:]
    
    # Build flyby dates
    flyby_dates = [time.Time(best_launch_jd, format="jd", scale="tdb")]
    for tof in best_tofs:
        flyby_dates.append(flyby_dates[-1] + tof * u.day)
    
    print(f"\n{'='*60}")
    print("Best VVEJSA Trajectory")
    print(f"{'='*60}")
    for i, (d, b) in enumerate(zip(flyby_dates, flyby_bodies)):
        label = "Launch" if i == 0 else ("Arrival" if i == len(flyby_dates)-1 else f"Flyby {i}")
        print(f"{label:12s} at {b.name:10s}: {d.iso}")
    print()
    for i, name in enumerate(leg_names):
        tof = (flyby_dates[i+1] - flyby_dates[i]).to(u.day)
        print(f"  Leg {i+1} ({name:18s}): {tof:.1f}")
    total_tof = (flyby_dates[-1] - flyby_dates[0]).to(u.day)
    print(f"\n  Total mission:          {total_tof:.1f} ({total_tof.to(u.yr):.2f})")
    print(f"  Launch C3:              {best_c3:.2f} km²/s²")
    print(f"  Total V∞ mismatch:      {best_mismatch:.3f} km/s")

## Detailed Analysis of Best Trajectory
Rebuild Lambert solutions and compute gravity assist parameters.

In [ ]:
# Rebuild full Lambert solutions for the best trajectory
transfer_orbits = []
departure_dvs = []

print("Lambert Transfer Solutions")
print("=" * 70)
for i in range(5):
    dep_orb = Orbit.from_ephem(Sun, ephem_map[flyby_bodies[i]], flyby_dates[i])
    arr_orb = Orbit.from_ephem(Sun, ephem_map[flyby_bodies[i+1]], flyby_dates[i+1])
    man_lambert = Maneuver.lambert(dep_orb, arr_orb)
    trans_orb = dep_orb.apply_maneuver(man_lambert, intermediate=True)[0]
    transfer_orbits.append(trans_orb)
    
    dv_mag = np.linalg.norm(man_lambert[0][1].value)
    departure_dvs.append(dv_mag)
    tof = (flyby_dates[i+1] - flyby_dates[i]).to(u.day)
    print(f"Leg {i+1} ({leg_names[i]:18s}): TOF = {tof:8.1f}, DV = {dv_mag:.3f} km/s")

print(f"\nLaunch C3: {departure_dvs[0]**2:.2f} km²/s²")

# Gravity assist analysis
def compute_gravity_assist(body, body_ephem, flyby_date,
                           incoming_transfer, outgoing_transfer):
    body_orb = Orbit.from_ephem(Sun, body_ephem, flyby_date)
    body_vel = body_orb.rv()[1]
    incoming_at_flyby = incoming_transfer.propagate(flyby_date)
    v_arr_helio = incoming_at_flyby.rv()[1]
    v_dep_helio = outgoing_transfer.rv()[1]
    
    v_inf_in = v_arr_helio - body_vel
    v_inf_out = v_dep_helio - body_vel
    v_inf_in_mag = np.linalg.norm(v_inf_in).to(u.km / u.s)
    v_inf_out_mag = np.linalg.norm(v_inf_out).to(u.km / u.s)
    
    cos_turn = np.dot(v_inf_in.value, v_inf_out.value) / (
        np.linalg.norm(v_inf_in.value) * np.linalg.norm(v_inf_out.value))
    cos_turn = np.clip(cos_turn, -1, 1)
    turn_angle = np.arccos(cos_turn)
    turn_angle_deg = np.degrees(turn_angle)
    
    v_inf_avg = (v_inf_in_mag + v_inf_out_mag) / 2
    sin_half = np.sin(turn_angle / 2)
    if sin_half > 0:
        r_p = (body.k / (v_inf_avg**2) * (1/sin_half - 1)).to(u.km)
    else:
        r_p = np.inf * u.km
    altitude = (r_p - body.R).to(u.km)
    
    return {
        'body': body, 'v_inf_in': v_inf_in_mag, 'v_inf_out': v_inf_out_mag,
        'v_inf_diff': abs(v_inf_out_mag - v_inf_in_mag),
        'turn_angle_deg': turn_angle_deg, 'periapsis_km': r_p,
        'altitude_km': altitude, 'body_radius_km': body.R.to(u.km),
    }

flyby_names = ["Venus 1", "Venus 2", "Earth", "Jupiter"]
assist_results = []

print("\nGravity Assist Analysis")
print("=" * 80)
for i in range(4):
    flyby_idx = i + 1
    result = compute_gravity_assist(
        flyby_bodies[flyby_idx], ephem_map[flyby_bodies[flyby_idx]],
        flyby_dates[flyby_idx], transfer_orbits[i], transfer_orbits[i + 1],
    )
    assist_results.append(result)
    feasible = "YES" if result['altitude_km'].value > 0 else "NO (below surface!)"
    print(f"\n{flyby_names[i]} flyby ({result['body'].name}):")
    print(f"  V∞ incoming:    {result['v_inf_in']:.3f}")
    print(f"  V∞ outgoing:    {result['v_inf_out']:.3f}")
    print(f"  |V∞| mismatch:  {result['v_inf_diff']:.3f}")
    print(f"  Turning angle:  {result['turn_angle_deg']:.2f}°")
    print(f"  Flyby altitude: {result['altitude_km']:.1f}")
    print(f"  Feasible:       {feasible}")

## Plot Best Trajectory

In [ ]:
plotter = StaticOrbitPlotter(dark=True)

# Plot planet orbits at encounter times
plotted_bodies = set()
for i, (body, date) in enumerate(zip(flyby_bodies, flyby_dates)):
    orb = Orbit.from_ephem(Sun, ephem_map[body], date)
    label = f"{body.name} orbit" if body.name not in plotted_bodies else None
    if label:
        plotter.plot(orb, label=label)
        plotted_bodies.add(body.name)

# Plot transfer orbits
for i, trans in enumerate(transfer_orbits):
    plotter.plot(trans, label=leg_names[i])

ax = getattr(plotter, "ax", None) or plotter._ax
plot_range = 1.3e9
ax.set_xlim(-plot_range, plot_range)
ax.set_ylim(-plot_range, plot_range)
ax.set_aspect("equal", adjustable="box")
ax.set_title("Best VVEJSA Trajectory Found")

plt.show()

# Inner solar system detail
plotter2 = StaticOrbitPlotter(dark=True)
venus_orb = Orbit.from_ephem(Sun, ephem_venus, flyby_dates[1])
earth_orb = Orbit.from_ephem(Sun, ephem_earth, flyby_dates[0])
plotter2.plot(venus_orb, label="Venus orbit")
plotter2.plot(earth_orb, label="Earth orbit")
for i in range(3):
    plotter2.plot(transfer_orbits[i], label=leg_names[i])

ax2 = getattr(plotter2, "ax", None) or plotter2._ax
inner_range = 3e8
ax2.set_xlim(-inner_range, inner_range)
ax2.set_ylim(-inner_range, inner_range)
ax2.set_aspect("equal", adjustable="box")
ax2.set_title("Inner Solar System Flybys")

plt.show()

# Print summary
print("\nVVEJSA Trajectory Summary")
print("=" * 60)
print(f"Launch Date:    {flyby_dates[0].iso}")
print(f"Saturn Arrival: {flyby_dates[-1].iso}")
total_tof = (flyby_dates[-1] - flyby_dates[0]).to(u.day)
print(f"Total TOF:      {total_tof:.1f} ({total_tof.to(u.yr):.2f})")
print(f"Launch C3:      {departure_dvs[0]**2:.2f} km²/s²")
print(f"Launch V∞:      {departure_dvs[0]:.3f} km/s")

print("\n--- Flyby Summary ---")
for i, (name, res) in enumerate(zip(flyby_names, assist_results)):
    print(f"{name:10s}: V∞={res['v_inf_in']:.2f} → {res['v_inf_out']:.2f}, "
          f"turn={res['turn_angle_deg']:.1f}°, alt={res['altitude_km']:.0f}")

print("\n--- Leg Details ---")
for i in range(len(leg_names)):
    tof = (flyby_dates[i+1] - flyby_dates[i]).to(u.day)
    print(f"{leg_names[i]:18s}: {tof:8.1f}")